# 01 – Cài đặt CatVTON trên Colab / Kaggle (PV1, việc 1.1)

**Vì sao cần notebook này:** Python mặc định của Colab/Kaggle là 3.13, còn CatVTON cần **Python 3.9** (`torch==2.4.0` không có bản cho 3.13; repo CatVTON kèm file biên dịch `detectron2/_C.cpython-39-*.so`).
Notebook tạo một môi trường Python 3.9 riêng bằng `uv` và cài các gói đã ghim trong `requirements-model.txt`. Chạy từ trên xuống; lỗi nào thì xem `docs/pv1_troubleshooting.md`.

Colab: *Runtime → Change runtime type → T4 GPU*. Kaggle: bật *Accelerator = GPU* và *Internet = On* (cần xác minh số điện thoại).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive, lấy code của nhóm và CatVTON

In [ ]:
import os
from google.colab import drive          # Kaggle: bỏ 2 dòng drive, đặt TRYON_ROOT=/kaggle/working/tryon_project
drive.mount('/content/drive')
os.environ['TRYON_ROOT'] = '/content/drive/MyDrive/tryon_project'     # thư mục Drive chung (xem docs/handoff.md)
os.environ['HF_HOME'] = '/content/hf_cache'                            # cache mô hình trên đĩa Colab, không ghi vào Drive
os.environ['MPLBACKEND'] = 'Agg'      # Colab đặt backend inline mà môi trường 3.9 không có -> matplotlib lỗi
REPO = '/content/AI-Business-Project'

In [ ]:
%%bash
set -e
[ -d /content/AI-Business-Project ] || git clone https://github.com/chauthao1603-dotcom/AI-Business-Project.git /content/AI-Business-Project
cd /content/AI-Business-Project
git pull --ff-only || true
[ -d CatVTON ] || git clone --depth 1 https://github.com/Zheng-Chong/CatVTON.git CatVTON
ls

## 2. Tạo môi trường Python 3.9 và cài thư viện (mất vài phút)

In [ ]:
%%bash
set -e
pip -q install uv
uv venv --python 3.9 /content/venv39
cd /content/AI-Business-Project
uv pip install --python /content/venv39/bin/python -r requirements-model.txt
/content/venv39/bin/python --version

## 3. Kiểm tra: GPU, phiên bản thư viện, import CatVTON

In [ ]:
!/content/venv39/bin/python -c "import sys, torch, diffusers, transformers; print(sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| diffusers', diffusers.__version__, '| transformers', transformers.__version__)" 

In [ ]:
# import phần mô hình và phần tạo mask (DensePose/SCHP). Nếu lỗi ở đây, chép NGUYÊN VĂN lỗi gửi nhóm.
!cd /content/AI-Business-Project/CatVTON && /content/venv39/bin/python -c "import model.pipeline, model.cloth_masker; print('import CatVTON OK')" 

## 4. Chạy thử một cặp ảnh (việc 1.2 và 1.3)

Lần đầu mất vài phút vì tải checkpoint (SD1.5 inpainting + CatVTON + DensePose/SCHP, vài GB). Cần 1 ảnh người và 1 ảnh áo (lấy từ `data/raw/test/` trên Drive, hoặc tải tạm 2 ảnh bất kỳ lên `/content`).

In [ ]:
PERSON = os.environ['TRYON_ROOT'] + '/data/raw/test/image/00006_00.jpg'     # đổi tên file cho khớp dữ liệu thật
GARMENT = os.environ['TRYON_ROOT'] + '/data/raw/test/cloth/00012_00.jpg'
!cd {REPO} && /content/venv39/bin/python -m scripts.smoke_tryon --person "{PERSON}" --garment "{GARMENT}" --out /content/smoke

# Nếu bước import cloth_masker ở mục 3 báo lỗi (DensePose), vẫn thử mô hình được bằng mask sẵn của VITON-HD:
# MASK = os.environ['TRYON_ROOT'] + '/data/raw/test/agnostic-mask/00006_00_mask.png'
# !cd {REPO} && /content/venv39/bin/python -m scripts.smoke_tryon --person "{PERSON}" --garment "{GARMENT}" --mask "{MASK}" --out /content/smoke

In [ ]:
from PIL import Image
import glob
f = sorted(glob.glob('/content/smoke/*.png'))
[print(x) for x in f]
display(Image.open([x for x in f if not x.endswith('_mask.png')][0]))

**Xong khi:** thấy đúng người mặc đúng áo, dòng in ra có *giây/ảnh* và *VRAM đỉnh* (ghi vào báo cáo). Lưu `notebooks/01_setup.ipynb` bản đã chạy (xóa output ảnh trước khi commit) và cập nhật `requirements-model.txt` nếu phải đổi phiên bản gói.

Chạy Streamlit trong cùng môi trường (việc 1.6): `!/content/venv39/bin/streamlit run app.py` rồi mở ra ngoài bằng `cloudflared` (xem `docs/PHAN_VIEC_v2.md`).

## 5. Kiểm tra 3 cặp ảnh khác nhau (hoàn tất việc 1.3)

Chạy `run_tryon` trên 3 cặp trong CÙNG một tiến trình, kiểm tra tự động chuẩn đầu ra (ảnh 768×1024 RGB, mask 768×1024 chỉ gồm 0/255). Cặp đầu tiên chậm hơn vì gồm cả thời gian nạp mô hình.

In [ ]:
%%bash
cd /content/AI-Business-Project
/content/venv39/bin/python - <<'EOF'
import os, time
import numpy as np, torch
from PIL import Image
from src import config
from src.tryon import run_tryon

CAT = str(config.CATVTON_DIR) + '/resource/demo/example'
pairs = [('/person/women/049713_0.jpg', '/condition/upper/21514384_52353349_1000.jpg'),
         ('/person/women/model_8.png', '/condition/upper/22790049_53294275_1000.jpg'),
         ('/person/men/model_5.png', '/condition/upper/23255574_53383833_1000.jpg')]
os.makedirs('/content/smoke3', exist_ok=True)
for i, (p, g) in enumerate(pairs, 1):
    t0 = time.time()
    res, mask = run_tryon(Image.open(CAT + p), Image.open(CAT + g))
    dt = time.time() - t0
    m = np.asarray(mask)
    assert res.size == (768, 1024) and res.mode == 'RGB', (res.size, res.mode)
    assert mask.size == (768, 1024) and mask.mode == 'L' and set(np.unique(m)) <= {0, 255}
    res.save(f'/content/smoke3/{i}.png'); mask.save(f'/content/smoke3/{i}_mask.png')
    print(f'cặp {i}: {dt:.1f}s | vùng mask {100 * (m == 255).mean():.1f}% ảnh')
print('VRAM đỉnh:', round(torch.cuda.max_memory_allocated() / 1e9, 2), 'GB | GPU:', torch.cuda.get_device_name(0))
print('3 cặp đạt chuẩn đầu ra')
EOF

In [ ]:
from PIL import Image
import glob
fs = sorted(f for f in glob.glob('/content/smoke3/*.png') if not f.endswith('_mask.png'))
imgs = [Image.open(f).resize((384, 512)) for f in fs]
sheet = Image.new('RGB', (384 * len(imgs), 512), 'white')
for i, im in enumerate(imgs): sheet.paste(im, (384 * i, 0))
display(sheet)

## 6. Lưu ảnh mẫu lên Drive để gửi nhóm

In [ ]:
%%bash
mkdir -p "$TRYON_ROOT/outputs/smoke"
cp /content/smoke/*.png /content/smoke3/*.png "$TRYON_ROOT/outputs/smoke/"
ls "$TRYON_ROOT/outputs/smoke" 

## 7. Chạy hàng loạt 200 cặp (việc 1.4)

**Điều kiện:** PV2 đã đưa lên Drive thư mục `data/raw/test/` (có `image/`, `cloth/`, `agnostic-mask/`) và đã merge `data/processed/eval_pairs.csv` vào Git (rồi `git pull` trong bản clone, hoặc upload file đó vào thư mục `data/processed/` của bản clone).

Nhật ký đặt trên Drive (`--log_dir`) để không mất khi Colab ngắt phiên. Luôn thử `--limit 5` trước. Chạy lại cùng lệnh sẽ tự bỏ qua cặp đã có ảnh.

In [ ]:
LOGS = os.environ['TRYON_ROOT'] + '/results'
!cd {REPO} && /content/venv39/bin/python -m scripts.run_batch --limit 5 --log_dir "{LOGS}"

In [ ]:
# Sau khi 5 cặp thử đạt (xem ảnh trong Drive: outputs/catvton/), chạy toàn bộ (mất khoảng 200 × giây/ảnh; có thể chia nhiều phiên):
# !cd {REPO} && /content/venv39/bin/python -m scripts.run_batch --log_dir "{LOGS}"